# Testing Parallel OT
OT has been quite slow thus far. To speed this up, we've implemented a version of the main deviance-guided OT function that should parallelize the work. Here, we'll test if this has identical results to the main function and how much faster it is.

## Imports

In [1]:
### Enabling autoreload ##
%load_ext autoreload
%autoreload 2

In [2]:
### Directories and Files ###
root = '../../../../../'
metadata_dir = f"{root}Data/sc_data/cell_types.tsv"
data_dir = '/Users/oliviersmeets/Desktop/University/Master Internship 1/Data Handling/Data/sc_data' # Importing was not working with relative root+'..' approach, so replace with local data
save_dir = f"{root}Generated Data/Single-to-single/sc_uot_clustering/OT/DevianceSelection"

In [3]:
### Imports ###
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from umap import UMAP
from time import perf_counter
sys.path.append(root+'Scripts')
from hicdatautils import hic_ot_bulk_deviance_parallel, subset_clr_data, hic_ot_bulk_deviance

/opt/anaconda3/envs/hicotenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Pre-importing metadata ###
metadata_df = pd.read_csv(metadata_dir, sep="\t")
metadata_df

,cell_name,cell_type
0,arc_pair_7,T cells
1,arc_pair_18,CD4+ T cells
2,arc_pair_31,CD4+ T cells
3,arc_pair_56,T cells
4,arc_pair_57,CD8+ T cells
...,...,...
8058,arc_pair_736107,T cells
8059,arc_pair_736132,CD8+ T cells
8060,arc_pair_736144,CD8+ T cells
8061,arc_pair_736230,Monocytes


## Helper Functions

In [5]:
def generate_ot_data(
        reg_m: float = 0.1, 
        top_number : int = 250,
        count: int=20, 
        seed: int=42,
        optimize: bool=True
        ) -> tuple[pd.DataFrame, float]:
    '''
        Generates pair-wise OT comparison data for the immune cell data
        using the provided arguments. Additionally, keeps track of computation
        time.

        Function is specific to this notebook.

        Parameters
        ----------
        reg_m : float
            Unbalanced parameter to use.
            Default = 10.
        top_number : int
            Top number of contacts to use based on deviance.
            Default = 250.
        cell_count : int
            The top number of cells to be used.
            Default = 20.
        seed : int
            Seed for subsetting
            Default = 42.
        optimize : bool
            Determines whether or not to use the optimized function.
            Default = True

        Returns
        -------
        ot_results : pd.DataFrame
            Pandas dataframe with the desired results.
    '''
    # Subsetting
    clrs = subset_clr_data(data_dir, metadata_df, count, seed)
    
    # OT
        # Starting timer
    start_time = perf_counter()

        # Calculations
    if optimize:
        ot_results = hic_ot_bulk_deviance_parallel(
            clrs,
            "chr1",
            top_number,
            "max",
            reg_m
            )
    else:
        ot_results = hic_ot_bulk_deviance(
            clrs,
            "chr1",
            top_number,
            "max",
            reg_m
            )
    
        # Stopping timer
    end_time = perf_counter() - start_time

        # Saving results
    if optimize:
        ot_results.to_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_PARALLEL_unbalanced_reg_m={reg_m}_cluster_top={top_number}_count={count}_seed={seed}.csv')
    else:
        ot_results.to_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_unbalanced_reg_m={reg_m}_cluster_top={top_number}_count={count}_seed={seed}.csv')

    return ot_results, end_time

## Generating Data

In [ ]:
_, time_optim = generate_ot_data(0.1, 250, 40, 42)
_, time_default = generate_ot_data(0.1, 250, 40, 42, False)

/Users/oliviersmeets/Desktop/University/Master Internship 1/Data Handling/Testing/HiCOT Experimenting/Single Cell/UOT/Immune Cells/../../../../../Scripts/hicdatautils/hicgeneral.py:125: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
  9%|▉         | 3856/42778 [00:10<01:35, 408.98it/s]/opt/anaconda3/envs/hicotenv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
100%|██████████| 42778/42778 [01:51<00:00, 383.32it/s]


## Loading Data and Validating Equavalence

In [23]:
optim = pd.read_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_PARALLEL_unbalanced_reg_m=0.1_cluster_top=250_count=40_seed=42.csv', index_col=0)
default = pd.read_csv(f'{save_dir}/IMMUNE_CELLS_DEVIANCE_unbalanced_reg_m=0.1_cluster_top=250_count=40_seed=42.csv', index_col=0)
print(optim.equals(default))
print(f"Factor by which optimized method is faster: {time_default/time_optim:.2f}x")

True
Factor by which optimized method is faster: 3.84x


Outputs are identical